In [1]:
!pip install -q langchain langchain-community langchain-google-genai \
chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.

In [7]:
import os
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

os.environ["GOOGLE_API_KEY"] = api_key

print("Gemini API key added successfully!")

Enter your Gemini API key: ··········
Gemini API key added successfully!


In [8]:
document_text = """
Artificial Intelligence (AI)

Artificial Intelligence is a field of computer science that focuses
on creating systems capable of performing tasks that normally require
human intelligence.

Machine Learning

Machine Learning is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning

Deep Learning is a subset of Machine Learning that uses neural networks
with multiple layers. It is commonly used for image recognition,
speech recognition, and natural language processing.

Generative AI

Generative AI refers to AI systems that can create new content such as
text, images, audio, video, and computer code.

Large Language Models

Large Language Models (LLMs) are AI models trained on large amounts of
text data. They can understand and generate human-like text.

Retrieval Augmented Generation

Retrieval Augmented Generation (RAG) combines information retrieval
with language generation. A RAG system first retrieves relevant
information from a knowledge source and then provides that information
to a language model to generate a grounded answer.
"""

with open("ai_knowledge.txt", "w") as f:
    f.write(document_text)

print("Knowledge document created successfully!")

Knowledge document created successfully!


In [9]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("ai_knowledge.txt")

documents = loader.load()

print("Document loaded successfully!")
print("Number of documents:", len(documents))

Document loaded successfully!
Number of documents: 1


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Document split successfully!")
print("Number of chunks:", len(chunks))

Document split successfully!
Number of chunks: 3


In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Embedding model created successfully!")

Embedding model created successfully!


In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ai_knowledge"
)

print("Chroma vector database created successfully!")

Chroma vector database created successfully!


In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")

Retriever created successfully!


In [14]:
question = "What is Machine Learning?"

retrieved_docs = retriever.invoke(question)

print("Number of retrieved chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content)

Number of retrieved chunks: 3

--- Retrieved Chunk 1 ---
Artificial Intelligence (AI)

Artificial Intelligence is a field of computer science that focuses
on creating systems capable of performing tasks that normally require
human intelligence.

Machine Learning

Machine Learning is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning

--- Retrieved Chunk 2 ---
Deep Learning

Deep Learning is a subset of Machine Learning that uses neural networks
with multiple layers. It is commonly used for image recognition,
speech recognition, and natural language processing.

Generative AI

Generative AI refers to AI systems that can create new content such as
text, images, audio, video, and computer code.

Large Language Models

Large Language Models (LLMs) are AI models trained on large amounts of
text data. They can understand and generate human-like text.

--- Ret

In [15]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful question-answering assistant.

Answer the question using ONLY the context provided below.

If the answer is not present in the context, say:
"I don't have enough information in the provided document."

Context:
{context}

Question:
{question}

Answer:
""")

print("RAG prompt created successfully!")

RAG prompt created successfully!


In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("Gemini LLM is ready!")

Gemini LLM is ready!


In [18]:
question = "What is Machine Learning?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(final_prompt)

print("Question:", question)
print("\nAnswer:", response.content)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What is Machine Learning?

Answer: [{'type': 'text', 'text': 'Machine Learning is a subset of Artificial Intelligence that allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.', 'extras': {'signature': 'EoMJCoAJARFNMg+RnLAdnfyNEEdbXhrbkJ9EKYwbIj+k/5jp4DmqQuzE4atkTW5/JJo5dXa14rhdyDccr8s5t7YNi9DoDsI3Tj/sPF0pFjn3pmz+bF5SIYBoru1BJxAb1DlrpbWpPHzpVJyICuLsSHkWTveA2T/zR+gFSAnngMXAuNZD4wTzhNQLHXuiq1ek7JSefEJIvL8JIrm/pyrMkH2NtAQLI0GeVAzUZA0RmshFh3oBhchpg/nyIhWOydgx46V1+LqiPq/CdAoNFGsF11iZXFF4iuCaNcbTZYBMyjy5iLFMiRH2oJLMXhiE6gZaJNiRtbXVx/YDGdsX/mSC+wl1BGm9Lv7OT1zT8oXCh6jXD4q5XaVQyiRO2dp8zkt6D4J8Ajczcx+xVZXentoPDrWICXJR3cLtM/rlOgGh0RERUhwmgUXJ/qj8yWw9PUHve8GZcp1rkJ8yQn2Gtl3Ybad3XLT7ChiiVcALQBfly/1T3UnIAtv/VNUA4HWcWMvZSH9tnpn5pJsVZ2A4zohOoNya0MjSrV9COI7O82gu/H5Obpl6UeaqWZ91aY2dS0mvB996JThkyU9Cq6b3WYx83iJzOqsAzWYxURwupFRRKY+dQeNt2O1HMWqzu59kY7ZlK6v+47iKkLTsnsnedTww1fhpq/N0goUPgGudaYExP+GMHIXJZBWl9d0gGHjYDoFGVRcvI

In [19]:
question = "What is Deep Learning?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(final_prompt)

print("Question:", question)
print("\nAnswer:", response.content)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What is Deep Learning?

Answer: [{'type': 'text', 'text': 'Deep Learning is a subset of Machine Learning that uses neural networks with multiple layers. It is commonly used for image recognition, speech recognition, and natural language processing.', 'extras': {'signature': 'EqYICqMIARFNMg9u5MeLlWH4yPY0d9kfnpUfUhmqT8nstowRY+8UVVy2mLVXLEXP2xlxpwEiHAbMXargd0h5XRoWSKsm9Oo1QGnwOJlJmzTMr1Qzl36MOMN/rJGlJ/gKMg4WTUVPI4IaEMS7EwvN9APqgfIHKgrVgi/Fxdvni5/KeakmC8fAW5L8AGewgar1psbHPGO3yWoAPFiAqELCkBh/B9gSQ673m7b9DHuKnpqEZeKBl0rr4zFz38cEYIqS7/h6AgGXrhaJ68kxmJRRDHe53WchjJO4MX9j78v1EsH+njHpcAe9V5Pl+l6MhJSZ9guA0j73ozv+cTmCnp/w/FguS0vSj43tu7tqlGyf7vQUU19q+VL2X1hdHZO2EldymNRVXqqbBNFCUV72KxMblufmlfcaybomQ9xdfmkSf8+iYik2qmJcek30hqCj8M21ynszAR20H6y4wMRHST22hOZUN3zdAVKHlaLGqIlCTjJESYPpHe+k0WaMTHITycQhDqzLD90BUFRj8R5mZjpedYXymwCVZNQCOyxMIUmQNyASyFIChcMJ80M7bzGdTGS6q2esR5udMV99DAxR2YPStaRq9M7n7y6RdmKYq+yvEKdEQpy4dzaoxJtKySK0KrtLQSIcjKH94UUrpu034/3Qhqzt31/zw9eUDiMN3R0yUSv0FmnHRAQWTt04jdEpjnv561Sk7tGgYk

In [20]:
question = "What is the capital of France?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(final_prompt)

print("Question:", question)
print("\nAnswer:", response.content)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What is the capital of France?

Answer: [{'type': 'text', 'text': "I don't have enough information in the provided document.", 'extras': {'signature': 'ErQICrEIARFNMg+xmWZYOrtS53Kvw6SeWfnrGyNMEzNAK46Y8rdFD5wBrOQwkWpO75gpxl+0OdfzQvjP4J62WOvccDyGcUTiHQiNpZGn+C1OrkSOMhmtu8AAGZLoqh79kmy0yqVS9NaPThTTADj5Khj9Qz7oyimXMhZqLceMr8WWgo78vGMjEC7MW39e9Bqi9329/dg1KUWyHM70oPEB12oQIuOMPzD3JaPDTajaaVYHh1ifims7sfAdlnVWT0cLseo4eHpY3uF989eXbmqH8VKh/Fod+XmC7sbtQWSQ6UbbgctTapUldv77LuOTlCGuCUSvfdVbMU4Dr7Rrelv8TrWxGM7vJBF/jfP35kiTwHZkFB1mRwaKyBymF4b8/MiFisW7aAmDiIFy98OtqVP44gzm82ZyxVwCDyTap/1kDqUESKtfj7JzEOyDzWf5jEesVZlEdV+QF6HiP44kN6MuU3sO73CzSShb8dLdsoIQq+oKqgquDL4M6PlVO/8ExGS2RPw1N3/dxRi8JY1lOw7ghAxsQqiAAIqZUsFAfUVZdQc1JY7YBd2TbuUS9PqLjTAeLbRMKcQsnrUdvs2xz8AcXEBZHgJGeRKuxpOL0M2v+bPzp1iau7Bk1eXgaJHs8ymoOjQD8N4QQJSR7N24UVR5SqkSdhR/QJrn8FyeCm/gXqnN5Dk6Qjv9dX7O2VrxehR+6eHk0x1MUH8TSVjoIFuqw3yvg9wfjyELLCpTCJZguK/Gfebkae+D07KE+qS2F5B8QR0NnAZeKohMZhaTYSIUGZKmd37w2mMPlqG3nwArhCTamUl+zOdXv+ly3yWD9U5QJNH2kg

In [21]:
question = "What is Generative AI?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(final_prompt)

print("Question:", question)
print("\nAnswer:", response.text)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What is Generative AI?

Answer: Generative AI refers to AI systems that can create new content such as text, images, audio, video, and computer code.


In [22]:
question = "What is Generative AI?"

retrieved_docs = retriever.invoke(question)

print("QUESTION:")
print(question)

print("\nRETRIEVED INFORMATION:")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)

QUESTION:
What is Generative AI?

RETRIEVED INFORMATION:

--- Chunk 1 ---
Deep Learning

Deep Learning is a subset of Machine Learning that uses neural networks
with multiple layers. It is commonly used for image recognition,
speech recognition, and natural language processing.

Generative AI

Generative AI refers to AI systems that can create new content such as
text, images, audio, video, and computer code.

Large Language Models

Large Language Models (LLMs) are AI models trained on large amounts of
text data. They can understand and generate human-like text.

--- Chunk 2 ---
Retrieval Augmented Generation

Retrieval Augmented Generation (RAG) combines information retrieval
with language generation. A RAG system first retrieves relevant
information from a knowledge source and then provides that information
to a language model to generate a grounded answer.

--- Chunk 3 ---
Artificial Intelligence (AI)

Artificial Intelligence is a field of computer science that focuses
on creating s

In [25]:
def ask_rag(question):
    # 1. Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # 2. Combine retrieved chunks
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # 3. Create the RAG prompt
    final_prompt = prompt.invoke({
        "context": context,
        "question": question
    })

    # 4. Generate answer using Gemini
    response = llm.invoke(final_prompt)

    # 5. Display result
    print("Question:", question)
    print("Answer:", response.text)

In [26]:
ask_rag("What is Machine Learning?")

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What is Machine Learning?
Answer: Machine Learning is a subset of Artificial Intelligence that allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.
